#**Purpose**

This notebook runs inference on the fine tuned `bart-base` model for question generation.

Model Trained on: **Kaggle**

## **Install the required modules**

In [1]:
!pip install -qU \
 transformers

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.1/12.1 MB 59.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 67.1 MB/s eta 0:00:00


**Restart the session after all the modules are installed.**

##**Import the required libraries**

In [1]:
import torch
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM, pipeline

##**Initialize the Model from Huggingface**

In [2]:
MODEL_REPO = "gaurav-dey/bart-base-qg"
MAX_INPUT_LENGTH = 512                  # context+answer prompt length
MAX_TARGET_LENGTH = 96
NUM_BEAMS=4

In [3]:
device = "cuda" if torch.cuda.is_available() else "cpu"

In [4]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_REPO)
model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_REPO).to(device)

config.json:   0%|          | 0.00/1.62k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/416 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/3.56M [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  558MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/260 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/323 [00:00<?, ?B/s]

In [5]:
def generate_question(context, answer):
  prompt = (
      f"Target Answer: {answer}\n"
      f"Generate a question from the following context where the target answer is the correct answer. "
      f"Do not include phrases like 'According to the text' in the question and do not repeat the context in the question.\n"
      f"Context: {context}"
  )

  input_ids = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=MAX_INPUT_LENGTH).input_ids.to(device)
  output_ids = model.generate(input_ids, max_length=MAX_TARGET_LENGTH, num_beams=NUM_BEAMS)
  question = tokenizer.decode(output_ids[0], skip_special_tokens=True)

  return question

In [6]:
context = "The Eiffel Tower is a wrought-iron lattice tower on the Champ de Mars in Paris, France. It is named after the engineer Gustave Eiffel, whose company designed and built the tower."
answer = "Gustave Eiffel"

question = generate_question(context, answer)
question

'Who designed and built the Eiffel Tower?'

##**Huggingface Pipeline no longer supported for E-D Models in Transformers 5.X.X**

Note that we can not use a huggingface pipeline with encoder-decoder models like T5/Flan-T5/BART anymore as it is deprecated in transformers version 5 and above.

In [7]:
import transformers
transformers.__version__

'5.16.1'